# How to select products contaminated with RFI

### 1. Packages and Libraries

In [1]:
%pip install aiohttp requests
%load_ext autoreload
%autoreload 2
    
from shapely.geometry import shape, mapping
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from tqdm import tqdm
import requests
import pathlib
import folium
import os
import json
import time
import antimeridian
import yaml
from pathlib import Path

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load the config.yaml to access the paths
cfg_file = Path("../config.yaml").resolve()
cfg = yaml.safe_load(cfg_file.read_text())

# Set the name of the path in the config.yaml
token_path = (cfg_file.parent / cfg["paths"]["token_file"]).resolve()
out_path = (cfg_file.parent / cfg["paths"]["output"]).resolve()

rfi_prod_path = out_path / cfg["files"]["rfi_products"]
gdf_path = out_path / cfg["files"]["gdf"]

print(f"Token Path: {token_path}")
print(f"Output Path: {out_path}")

Token Path: /home/amelis/projects/BIOMASS/token.txt
Output Path: /home/amelis/projects/BIOMASS/biomass-hackathon-2026/topics/02-rfi-removal/out


### 2. Query the catalog

Query a large amount of products to identify those affected by RFI. The typical filters of STAC can be used (by location, time, ids, etc.). Working with L1A data is prefered.

#### 2.1 Search in the catalog

In [4]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

bbox_antarctica    = [-180, -90, 180, -60]
bbox_asia = [25.0, -11.0, 180.0, 82.0]

search = catalog.search(
    collections = ["BiomassLevel1a"],
    datetime = ['2026-01-01T00:00:00Z','2026-07-01T23:59:59Z'],
    bbox = bbox_asia,
    max_items = 2000,
    limit = 2000,
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

KeyboardInterrupt: 

#### 2.2 Inspect the searched items

Create the GeoDataframe to plot the footprints of the selected items

In [3]:
features = []
for it in items:
    if it.geometry is None:
        raise ValueError(f"Item {it.id} has no geometry (footprint).")
    props = it.properties or {}
    features.append({
        "type": "Feature",
        "geometry": it.geometry,
        "properties": {
            "product_id": it.id,
            "start_datetime": props.get("start_datetime"),
            "end_datetime": props.get("end_datetime"),
        },
    })


gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf = gdf.reset_index()
gdf = gdf[["index", "product_id", "start_datetime", "end_datetime", "geometry"]]
gdf["geometry"] = gdf.geometry.apply(lambda g: shape(antimeridian.fix_shape(mapping(g))))
gdf.head(len(items))

NameError: name 'items' is not defined

In [5]:
# Save the gdf for plotting the footprints on subsequent notebooks
gdf.to_parquet(gdf_path)
print(len(gdf))

2000


#### 2.3 Plot the footprints

In [8]:
## Ensure both are WGS84 for folium
gdf = gdf.to_crs("EPSG:4326")

# Center/zoom using bounds
bounds = gpd.GeoSeries(list(gdf.geometry), crs="EPSG:4326")
minx, miny, maxx, maxy = bounds.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=3)

def style(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.10}  

GeoJson(
    gdf.__geo_interface__,
    name=f"Products ({len(gdf)})",
    tooltip=folium.GeoJsonTooltip(fields=["product_id", "index"]),
    style_function=style,
).add_to(m)


folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m

### 3 Select products with RFI

Stream the annotation file of the selected items and search for the rfiDetectionFlag. If true, the product is affected by RFI. If so, check the rfiCorrectionFlag which is up when mitigation is applied, otherwise the RFI is intact. This is driven by the geospatial map (RFI mitigation applied on desert areas and Asia).

Save the RFI-related attributes:
- Percentage of bandwidth affected
- Percentage of lines affected
- Max bandwidth

Only the stats related to the Frequency Domain Notch Filter are selected because if RFI is detected on with the Time Domain algorithm it will also affect the time domain given the nature of RFIs in this band.

Additionally, due to the nature of the detection algorithm implemented in the BPS, it is possible to have a large False Alarm rate, meaning many products will be identified as RFI-affected but they might not. For that reason, all stats are saved so the user is able to filter by min percentage of affected bandwidth and/or a min percentage of affected lines to avoid false alarms.

In [7]:
# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

Insert the path to your token file from the config.ymal

In [8]:
if pathlib.Path(token_path).exists():
  with open(token_path, "rt") as f:
    OFFLINE_TOKEN = f.read().strip().replace("\n","")
    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')
    print("Access token retrieved successfully.")

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

Access token retrieved successfully.


In [11]:
from explore_annotation import read_annot_file

In [12]:
# Parallelize the stream of the annotation files
import threading
from concurrent.futures import ThreadPoolExecutor
from aiohttp import ClientResponseError

tok = {"v": access_token}
lock = threading.Lock()

def refresh_access_token():
    r = requests.post(
        "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token",
        data={"client_id": CLIENT_ID, "client_secret": CLIENT_SECRET,
              "grant_type": "refresh_token", "refresh_token": OFFLINE_TOKEN,
              "scope": "offline_access openid"})
    r.raise_for_status()
    return r.json()["access_token"]

def annot_job(it, retries=3):
    """Returns (id, results, error). Never raises, so one bad product can't kill the batch."""
    for attempt in range(retries):
        used = tok["v"]
        try:
            if not it.assets or "enclosure_annot_xml" not in it.assets:
                raise ValueError(f"Item {it.id} has no annotation asset.")
            res = read_annot_file(it.id, it.assets["enclosure_annot_xml"].href, used)
            return it.id, res, None
        except ClientResponseError as e:
            if e.status in (401, 403):
                with lock:
                    if tok["v"] == used:            # only the first thread refreshes
                        tok["v"] = refresh_access_token()
            time.sleep(2 ** attempt)                # back off 1, 2, 4 s
            err = repr(e)
        except Exception as e:
            time.sleep(2 ** attempt)
            err = repr(e)
    return it.id, [], err

N_workers = 8 # Adapt to the number of physical cores of your machine

with ThreadPoolExecutor(max_workers=N_workers) as ex:
    results = list(tqdm(ex.map(annot_job, items), total=len(items)))

products_rfi = [r for _, res, err in results for r in res]
failed = {pid: err for pid, _, err in results if err}
print(len(products_rfi), "RFI products,", len(failed), "failed")

  0%|          | 1/200 [00:00<01:55,  1.72it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true


  2%|▏         | 3/200 [00:00<00:45,  4.36it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = true


  2%|▎         | 5/200 [00:01<00:34,  5.68it/s]

...

rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true


 98%|█████████▊| 197/200 [00:28<00:00,  5.24it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true
rfiDetectionFlag = true
rfiCorrectionFlag = true


100%|██████████| 200/200 [00:29<00:00,  6.68it/s]

rfiDetectionFlag = true
rfiCorrectionFlag = true
200 RFI products, 0 failed


In [13]:
# save, right after the products_rfi loop
with open(rfi_prod_path, "w") as f:
    json.dump(products_rfi, f, indent=2)